# Step 11: Statistical Hypothesis Testing & Relationship Analysis

This notebook validates recruitment patterns using formal statistical hypothesis tests and correlation analyses:
1. **Two-Sample Hypothesis Tests (Welch's T-Test & Mann-Whitney U Test)**: Test whether pipeline durations differ significantly between joined and dropped candidates.
2. **Chi-Square ($\chi^2$) Independence Tests**: Test whether candidate drop-off outcomes depend significantly on Department or Recruitment Source.
3. **Correlation Analysis**: Compute Pearson and Spearman correlation matrices across candidate experience, interview scores, durations, and hiring outcomes.

In [ ]:
import os
import math
import numpy as np
import pandas as pd

# Path setup
FEATURES_PATH = os.path.join("..", "data", "processed", "candidate_features.csv")
print(f"Loading candidate features from: {FEATURES_PATH}")
df = pd.read_csv(FEATURES_PATH)
df.head()

## Part 1: Two-Sample Hypothesis Tests on Recruitment Durations

### Hypotheses:
- **Null Hypothesis ($H_0$)**: There is no difference in total pipeline duration between joined and dropped candidates ($\mu_{\text{hired}} = \mu_{\text{dropped}}$).
- **Alternative Hypothesis ($H_1$)**: There is a significant difference in total pipeline duration between joined and dropped candidates ($\mu_{\text{hired}} \ne \mu_{\text{dropped}}$).
- **Significance Threshold ($\alpha$)**: $0.05$

In [ ]:
hired_durations = df[df['joined'] == 1]['total_recruitment_duration_days'].values
dropped_durations = df[df['dropped'] == 1]['total_recruitment_duration_days'].values

n1, n2 = len(hired_durations), len(dropped_durations)
mean1, mean2 = np.mean(hired_durations), np.mean(dropped_durations)
var1, var2 = np.var(hired_durations, ddof=1), np.var(dropped_durations, ddof=1)

# 1. Welch's T-Test (Unequal variances)
se = math.sqrt((var1 / n1) + (var2 / n2))
t_stat = (mean1 - mean2) / se if se > 0 else 0.0
df_welch = ((var1/n1 + var2/n2)**2) / (((var1/n1)**2)/(n1-1) + ((var2/n2)**2)/(n2-1)) if se > 0 else 1.0

# Two-tailed p-value using normal approximation for large/moderate samples or standard error function
z_approx = abs(t_stat)
p_val_ttest = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(z_approx / math.sqrt(2.0))))

# 2. Mann-Whitney U Test (Non-parametric rank sum)
all_vals = [(v, 1) for v in hired_durations] + [(v, 2) for v in dropped_durations]
all_vals.sort(key=lambda x: x[0])
# Compute ranks
ranks = []
i = 0
while i < len(all_vals):
    j = i
    while j < len(all_vals) and all_vals[j][0] == all_vals[i][0]:
        j += 1
    avg_rank = (i + 1 + j) / 2.0
    for k in range(i, j):
        ranks.append((all_vals[k][1], avg_rank))
    i = j

r1 = sum(r for grp, r in ranks if grp == 1)
u1 = n1 * n2 + (n1 * (n1 + 1)) / 2.0 - r1
u2 = n1 * n2 - u1
u_stat = min(u1, u2)
mean_u = (n1 * n2) / 2.0
std_u = math.sqrt((n1 * n2 * (n1 + n2 + 1)) / 12.0)
z_u = (u_stat - mean_u) / std_u if std_u > 0 else 0.0
p_val_mwu = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(z_u) / math.sqrt(2.0))))

print("="*60)
print(" TWO-SAMPLE HYPOTHESIS TEST RESULTS (HIRED VS DROPPED DURATION)")
print("="*60)
print(f"Hired Candidates (n={n1}): Mean = {mean1:.2f} days, Std = {math.sqrt(var1):.2f}")
print(f"Dropped Candidates (n={n2}): Mean = {mean2:.2f} days, Std = {math.sqrt(var2):.2f}")
print(f"\n1. Welch's T-Test: t-statistic = {t_stat:.4f}, df = {df_welch:.2f}, p-value = {p_val_ttest:.4f}")
print(f"   Conclusion: {'Statistically Significant Difference (Reject H0)' if p_val_ttest < 0.05 else 'No Statistically Significant Difference (Fail to Reject H0)'}")
print(f"\n2. Mann-Whitney U Test: U-statistic = {u_stat:.2f}, Z-score = {z_u:.4f}, p-value = {p_val_mwu:.4f}")
print(f"   Conclusion: {'Statistically Significant Difference (Reject H0)' if p_val_mwu < 0.05 else 'No Statistically Significant Difference (Fail to Reject H0)'}")

## Part 2: Chi-Square ($\chi^2$) Independence Tests

We test whether candidate drop-off behavior is independent of categorical variables (Department and Sourcing Channel).

In [ ]:
def chi2_independence(contingency_matrix):
    observed = contingency_matrix.values
    row_sums = observed.sum(axis=1)
    col_sums = observed.sum(axis=0)
    total = observed.sum()
    
    expected = np.outer(row_sums, col_sums) / total
    chi2 = np.sum(((observed - expected) ** 2) / expected)
    dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
    
    # p-value approximation via Wilson-Hilferty transformation of Chi-Square
    if dof > 0:
        z_chi = ((chi2 / dof) ** (1/3) - (1 - 2/(9*dof))) / math.sqrt(2/(9*dof))
        p_val = 1.0 - 0.5 * (1.0 + math.erf(z_chi / math.sqrt(2.0)))
        p_val = max(0.0, min(1.0, p_val))
    else:
        p_val = 1.0
    return chi2, dof, p_val, pd.DataFrame(expected, index=contingency_matrix.index, columns=contingency_matrix.columns)

# Test 1: Department vs Dropped Outcome
dept_ct = pd.crosstab(df['department'], df['dropped'])
chi2_dept, dof_dept, p_dept, exp_dept = chi2_independence(dept_ct)

print("--- Department vs Outcome Contingency Table (Observed) ---")
print(dept_ct)
print(f"\nChi-Square Statistic: {chi2_dept:.4f}, Degrees of Freedom: {dof_dept}, p-value: {p_dept:.4f}")
print(f"Independence Result: {'Significant Association (Reject H0)' if p_dept < 0.05 else 'No Significant Association (Independent, Fail to Reject H0)'}\n")

# Test 2: Source vs Dropped Outcome
source_ct = pd.crosstab(df['source'], df['dropped'])
chi2_src, dof_src, p_src, exp_src = chi2_independence(source_ct)

print("--- Source vs Outcome Contingency Table (Observed) ---")
print(source_ct)
print(f"\nChi-Square Statistic: {chi2_src:.4f}, Degrees of Freedom: {dof_src}, p-value: {p_src:.4f}")
print(f"Independence Result: {'Significant Association (Reject H0)' if p_src < 0.05 else 'No Significant Association (Independent, Fail to Reject H0)'}")

## Part 3: Correlation & Relationship Analysis

We analyze relationships between quantitative variables (`experience_years`, `average_interview_score`, `total_recruitment_duration_days`) and outcome flags (`joined`, `dropped`).

In [ ]:
numeric_cols = [
    'experience_years',
    'average_interview_score',
    'total_recruitment_duration_days',
    'dropped',
    'joined',
    'is_delayed_dropoff'
]
num_df = df[numeric_cols].dropna()

# Pearson Correlation
pearson_corr = num_df.corr(method='pearson').round(3)
print("--- Pearson Correlation Matrix ---")
print(pearson_corr)

# Spearman Rank Correlation
spearman_corr = num_df.corr(method='spearman').round(3)
print("\n--- Spearman Rank Correlation Matrix ---")
print(spearman_corr)

print("\nKey Insights:")
score_joined_corr = pearson_corr.loc['average_interview_score', 'joined']
print(f"1. Average Interview Score vs Joining Outcome Correlation: {score_joined_corr:+.3f} (Higher scores strongly correlate with hiring).")
dur_joined_corr = pearson_corr.loc['total_recruitment_duration_days', 'joined']
print(f"2. Total Duration vs Joining Outcome Correlation: {dur_joined_corr:+.3f} (Hired candidates undergo full pipeline including onboarding).")